In [0]:
#hiiiii
%sqlSHOW SCHEMAS IN <catalog_name_you_saw>;SHOW SCHEMAS IN <catalog_name_you_saw>;
show catalogs;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS emil_cat.bronze;
CREATE SCHEMA IF NOT EXISTS emil_cat.silver;
CREATE SCHEMA IF NOT EXISTS emil_cat.gold;

In [0]:
from pyspark.sql import Row
import random

customers_data = [
    Row(customer_id=1, customer_name="Acme Retail",   region="APAC", signup_date="2022-01-15"),
    Row(customer_id=2, customer_name="Blue Ocean Co", region="EMEA", signup_date="2022-03-02"),
    Row(customer_id=3, customer_name="Cedar Traders",  region="AMER", signup_date="2022-04-19"),
    Row(customer_id=4, customer_name="Delta Supplies",region="APAC", signup_date="2022-06-11"),
    Row(customer_id=5, customer_name="Echo Retail",   region="EMEA", signup_date="2022-08-25"),
    # intentional dirt: a duplicate row (same customer_id=2 re-sent by the source)
    Row(customer_id=2, customer_name="Blue Ocean Co", region="EMEA", signup_date="2022-03-02"),
    # intentional dirt: a null customer_id (a bad extract row)
    Row(customer_id=None, customer_name="Unknown Corp", region="AMER", signup_date="2023-01-01"),
]

df_customers = spark.createDataFrame(customers_data)
df_customers.write.mode("overwrite").saveAsTable("emil_cat.bronze.customers_raw")
display(spark.table("emil_cat.bronze.customers_raw"))

In [0]:
products_data = [
    Row(product_code="P-100", product_name="Widget A", category="Hardware", unit_price=25.50),
    Row(product_code="P-101", product_name="Widget B", category="Hardware", unit_price=40.00),
    Row(product_code="P-102", product_name="Gadget X", category="Electronics", unit_price=99.99),
    Row(product_code="P-103", product_name="Gadget Y", category="Electronics", unit_price=149.50),
    # intentional dirt: inconsistent code formatting from the source system (extra space / wrong case)
    Row(product_code="p-100 ", product_name="Widget A", category="Hardware", unit_price=25.50),
]

df_products = spark.createDataFrame(products_data)
df_products.write.mode("overwrite").saveAsTable("emil_cat.bronze.products_raw")
display(spark.table("emil_cat.bronze.products_raw"))

In [0]:
orders_data = [
    Row(order_id=101, customer_id=1, product_code="P-100", quantity=3, order_date="2025-08-22"),
    Row(order_id=102, customer_id=2, product_code="P-101", quantity=5, order_date="2024-11-12"),
    Row(order_id=103, customer_id=4, product_code="P-102", quantity=3, order_date="2026-04-03"),
    Row(order_id=104, customer_id=4, product_code="P-103", quantity=50, order_date="2025-07-05"),
    Row(order_id=105, customer_id=5, product_code="P-104", quantity=20, order_date="2025-08-09"),
    Row(order_id=101, customer_id=6, product_code="P-105", quantity=23, order_date="2025-08-22"),
    Row(order_id=106, customer_id=7, product_code="P-106", quantity=5, order_date="2025-08-22"),
    Row(order_id=107, customer_id=8, product_code="P-107", quantity=6, order_date="2025-08-22"),
    Row(order_id=107, customer_id=9, product_code="P-108", quantity=10, order_date="2025-08-22"),
    Row(order_id=108, customer_id=10, product_code="P-109", quantity=12, order_date="2025-08-22"),
    # add this row for requirement 3 — a null quantity
    Row(order_id=109, customer_id=1, product_code="P-100", quantity=None, order_date="2025-08-22"),
]

df_orders = spark.createDataFrame(orders_data)
df_orders.write.mode("overwrite").saveAsTable("emil_cat.bronze.orders_raw")
display(spark.table("emil_cat.bronze.orders_raw"))

In [0]:
%sql
SELECT DISTINCT o.customer_id
FROM emil_cat.bronze.orders_raw o
LEFT JOIN emil_cat.bronze.customers_raw c
  ON o.customer_id = c.customer_id
WHERE c.customer_id IS NULL;

In [0]:
orders_raw = spark.table("emil_cat.bronze.orders_raw")
customers_raw = spark.table("emil_cat.bronze.customers_raw")

# keep only orders whose customer_id has a match in customers_raw
orders_clean = orders_raw.join(customers_raw.select("customer_id"), on="customer_id", how="inner")

# keep only orders whose customer_id has NO match in customers_raw
orders_quarantine = orders_raw.join(customers_raw.select("customer_id"), on="customer_id", how="left_anti")

print("Clean orders:", orders_clean.count())
print("Quarantined orders:", orders_quarantine.count())

display(orders_clean)
display(orders_quarantine)

In [0]:
from pyspark.sql import functions as F

orders_clean.groupBy("order_id").count().filter(F.col("count") > 1).show()

In [0]:
customers_raw.groupBy("customer_id").count().filter(F.col("count") > 1).show()

In [0]:
customers_lookup = customers_raw.select("customer_id").dropDuplicates(["customer_id"])

orders_clean = orders_raw.join(customers_lookup, on="customer_id", how="inner")
orders_quarantine = orders_raw.join(customers_lookup, on="customer_id", how="left_anti")

print("Clean orders:", orders_clean.count())
print("Quarantined orders:", orders_quarantine.count())

In [0]:
orders_raw = spark.table("emil_cat.bronze.orders_raw")
customers_raw = spark.table("emil_cat.bronze.customers_raw")

customers_lookup = customers_raw.select("customer_id").dropDuplicates(["customer_id"])

orders_clean = orders_raw.join(customers_lookup, on="customer_id", how="inner")
orders_quarantine = orders_raw.join(customers_lookup, on="customer_id", how="left_anti")

print("Clean orders:", orders_clean.count())
print("Quarantined orders:", orders_quarantine.count())

In [0]:
orders_clean.filter(orders_clean.quantity.isNull()).show()

In [0]:
from pyspark.sql import functions as F

customers_lookup = (
    customers_raw.select("customer_id")
    .dropDuplicates(["customer_id"])
    .withColumnRenamed("customer_id", "lookup_customer_id")
)

orders_checked = orders_raw.join(
    customers_lookup,
    orders_raw.customer_id == customers_lookup.lookup_customer_id,
    how="left"
)

orders_checked = (
    orders_checked
    .withColumn("is_valid_customer", F.col("lookup_customer_id").isNotNull())
    .withColumn("is_valid_quantity", F.col("quantity").isNotNull())
    .withColumn(
        "quarantine_reason",
        F.when(~F.col("is_valid_customer") & ~F.col("is_valid_quantity"), "invalid_customer_and_null_quantity")
         .when(~F.col("is_valid_customer"), "invalid_customer_id")
         .when(~F.col("is_valid_quantity"), "null_quantity")
    )
)

orders_clean = (
    orders_checked
    .filter(F.col("is_valid_customer") & F.col("is_valid_quantity"))
    .drop("lookup_customer_id", "is_valid_customer", "is_valid_quantity", "quarantine_reason")
)

orders_quarantine = (
    orders_checked
    .filter(~(F.col("is_valid_customer") & F.col("is_valid_quantity")))
    .drop("lookup_customer_id", "is_valid_customer", "is_valid_quantity")
)

print("Clean:", orders_clean.count())
print("Quarantined:", orders_quarantine.count())
display(orders_quarantine)

In [0]:
from pyspark.sql import functions as F

orders_raw = spark.table("emil_cat.bronze.orders_raw")
customers_lookup = (
    spark.table("emil_cat.bronze.customers_raw")
    .select("customer_id").dropDuplicates(["customer_id"])
    .withColumnRenamed("customer_id", "lookup_customer_id")
)
products_lookup = (
    spark.table("emil_cat.silver.products_clean")
    .select("product_code").dropDuplicates(["product_code"])
    .withColumnRenamed("product_code", "lookup_product_code")
)

orders_checked = (
    orders_raw
    .join(customers_lookup, orders_raw.customer_id == customers_lookup.lookup_customer_id, how="left")
    .join(products_lookup, orders_raw.product_code == products_lookup.lookup_product_code, how="left")
    .withColumn("is_valid_customer", F.col("lookup_customer_id").isNotNull())
    .withColumn("is_valid_product", F.col("lookup_product_code").isNotNull())
    .withColumn("is_valid_quantity", F.col("quantity").isNotNull())
    .withColumn(
        "quarantine_reason",
        F.when(~F.col("is_valid_customer") & ~F.col("is_valid_product"), "invalid_customer_and_product")
         .when(~F.col("is_valid_customer"), "invalid_customer_id")
         .when(~F.col("is_valid_product"), "invalid_product_code")
         .when(~F.col("is_valid_quantity"), "null_quantity")
    )
)

is_fully_valid = F.col("is_valid_customer") & F.col("is_valid_product") & F.col("is_valid_quantity")

orders_clean = orders_checked.filter(is_fully_valid).drop(
    "lookup_customer_id", "lookup_product_code", "is_valid_customer", "is_valid_product", "is_valid_quantity", "quarantine_reason"
)
orders_quarantine = orders_checked.filter(~is_fully_valid).drop(
    "lookup_customer_id", "lookup_product_code", "is_valid_customer", "is_valid_product", "is_valid_quantity"
)

print("Clean:", orders_clean.count())
print("Quarantined:", orders_quarantine.count())

orders_clean.write.mode("overwrite").saveAsTable("emil_cat.silver.orders_clean")
orders_quarantine.write.mode("overwrite").saveAsTable("emil_cat.silver.orders_quarantine")

In [0]:
from pyspark.sql.types import DecimalType

orders_clean = spark.table("emil_cat.silver.orders_clean")
dim_products = spark.table("emil_cat.gold.dim_products").withColumn(
    "unit_price", F.col("unit_price").cast(DecimalType(10, 2))
)

fact_orders = (
    orders_clean.join(dim_products, on="product_code", how="inner")
    .withColumn("order_amount", (F.col("quantity") * F.col("unit_price")).cast(DecimalType(10, 2)))
    .select("order_id", "customer_id", "product_code", "quantity", "unit_price", "order_amount", "order_date")
)

fact_orders.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("emil_cat.gold.fact_orders")
display(fact_orders)

In [0]:
display(spark.table("emil_cat.silver.orders_quarantine"))